# Этап 3. Эксперименты с моделью

На этом этапе нужно обучить модель, которая предсказывает цену квартиры. В ноутбуке я разбираюсь,
что за данные получились после очистки и какая модель имеет смысл, а потом переношу рабочий код
в скрипты DVC-пайплайна.

План:

1. читаю таблицу `clean_flats_dataset` из личной БД (её собрал DAG `clean_flats_dataset` из первой части);
2. смотрю, что внутри данных;
3. выбираю признаки и целевую переменную;
4. делю данные на train и test;
5. считаю метрики для простого бейзлайна и для CatBoost и сравниваю их;
6. смотрю, на какие признаки опирается модель.

Итоговый код лежит в `scripts/` и собран в пайплайн (`dvc.yaml`, `params.yaml`).
Перед запуском ноутбука в папке `part2_dvc` должен быть файл `.env` с доступами к личной БД
(шаблон - `.env_template`).

In [ ]:
import os
from urllib.parse import quote_plus

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

pd.set_option('display.max_columns', 50)

## 1. Читаю данные из личной БД

Очищенный датасет лежит в личной БД в таблице `clean_flats_dataset`. Доступы беру из `.env`,
в коде их не пишу.

In [ ]:
load_dotenv()  # переменные из .env попадают в os.environ

host = os.environ['DB_DESTINATION_HOST']
port = os.environ['DB_DESTINATION_PORT']
db_name = os.environ['DB_DESTINATION_NAME']
user = os.environ['DB_DESTINATION_USER']
# пароль прогоняю через quote_plus: спецсимволы в нём ломают строку подключения
password = quote_plus(os.environ['DB_DESTINATION_PASSWORD'])

engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{db_name}')

data = pd.read_sql('select * from clean_flats_dataset', engine)
print('строк и колонок:', data.shape)
data.head()

## 2. Смотрю, что внутри

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
# после очистки пропусков быть не должно, и каждая квартира должна встречаться один раз
print('пропусков всего:', data.isna().sum().sum())
print('строк:', len(data), ', уникальных flat_id:', data['flat_id'].nunique())

In [ ]:
data['price'].hist(bins=50, figsize=(8, 4))
plt.title('Распределение цены')
plt.xlabel('цена, руб.')
plt.show()

Цена распределена неравномерно: недорогих квартир много, дорогих мало, у гистограммы длинный
правый хвост. Это пригодится дальше, когда буду выбирать основную метрику.

## 3. Признаки и целевая переменная

Целевая переменная - `price`, цена квартиры в рублях. Значит, это задача регрессии.

Три колонки с идентификаторами в признаки не беру:

- `id` - это просто номер строки в таблице, он появился при записи датасета и к цене отношения не имеет;
- `flat_id` - идентификатор квартиры, у каждой строки он свой. Модель может запомнить такой признак
  и показать отличное качество на обучающей выборке, а на новых квартирах от него не будет никакой
  пользы, потому что там будут совсем другие номера;
- `building_id` - идентификатор дома. Сам номер ничего не говорит о доме, а всё полезное про дом
  уже есть в отдельных колонках: год постройки, тип, координаты, высота потолков, этажность, лифт.

Категориальные признаки - `building_type_int` и три флага `is_apartment`, `studio`, `has_elevator`.
Тип дома записан числом, но это код, а не величина: тип 5 не больше типа 1 в пять раз, поэтому
его нужно кодировать, а не масштабировать. Перед кодированием перевожу категориальные колонки
в строки, чтобы значения из БД (bool или int) и значения из csv-файлов пайплайна выглядели одинаково.

Все остальные колонки числовые: площади, этаж, количество комнат, год постройки, координаты и так далее.
Эти же списки записаны в `params.yaml` и используются скриптами пайплайна.

In [ ]:
target_col = 'price'
drop_cols = ['id', 'flat_id', 'building_id']
cat_cols = ['building_type_int', 'is_apartment', 'studio', 'has_elevator']

X = data.drop(columns=drop_cols + [target_col])
y = data[target_col]

X[cat_cols] = X[cat_cols].astype(str)
num_cols = [col for col in X.columns if col not in cat_cols]

print('категориальные признаки:', cat_cols)
print('числовые признаки:', num_cols)

## 4. Делю данные на train и test

Откладываю 20% строк на тест: на них модель не учится, поэтому по ним честнее видно качество.
`random_state` фиксирую, чтобы разбиение повторялось при каждом запуске. Те же два числа лежат
в `params.yaml` и используются в `scripts/split.py`.

In [ ]:
test_size = 0.2
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
print('train:', X_train.shape, 'test:', X_test.shape)

## 5. Какие метрики считаю

Основная метрика - **MAE**, средняя абсолютная ошибка. Почему именно она:

- MAE измеряется в рублях, поэтому её легко объяснить: в среднем модель ошибается на столько-то рублей;
- цена распределена с длинным правым хвостом, и несколько очень дорогих квартир сильно тянут RMSE вверх.
  MAE считает все ошибки с одинаковым весом, поэтому пара дорогих объектов не перекашивает оценку;
- по ней удобно сравнивать модели друг с другом и с бейзлайном.

Дополнительно смотрю ещё три метрики:

- **RMSE** - сильнее штрафует крупные промахи, на неё же обучается CatBoost (`loss_function: RMSE`);
- **MAPE** - относительная ошибка в процентах, удобна, когда квартиры стоят по-разному;
- **R2** - какую долю разброса цены объясняет модель. У предсказания константой R2 около нуля,
  так что по нему сразу видно, есть ли от модели толк.

## 6. Бейзлайн: медианная цена

Самая простая модель - предсказывать всем квартирам одно и то же число, медиану цены по обучающей
выборке. Если нормальная модель не обгонит бейзлайн, значит она ничего полезного не выучила.
Медиану беру, а не среднее, потому что из-за дорогих квартир среднее завышено.

In [ ]:
baseline = DummyRegressor(strategy='median')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

baseline_metrics = {
    'mae': round(mean_absolute_error(y_test, baseline_pred), 2),
    'rmse': round(mean_squared_error(y_test, baseline_pred) ** 0.5, 2),
    'mape': round(mean_absolute_percentage_error(y_test, baseline_pred), 4),
    'r2': round(r2_score(y_test, baseline_pred), 4),
}
baseline_metrics

## 7. Базовая модель CatBoost

Собираю пайплайн из двух частей. Сначала `ColumnTransformer`: категориальные колонки кодирует
`OneHotEncoder` (для флагов из двух значений хватает одного столбца, поэтому `drop='if_binary'`;
`handle_unknown='ignore'` нужен на случай, если в тесте встретится категория, которой не было
в обучении), числовые колонки приводит к одному масштабу `StandardScaler`. Дальше идёт
`CatBoostRegressor`. Точно такой же пайплайн собирается в `scripts/fit.py`, а его параметры
вынесены в `params.yaml`.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols),
])

model = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                          loss_function='RMSE', verbose=0, thread_count=-1,
                          random_seed=random_state)

pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
pipeline.fit(X_train, y_train)

catboost_pred = pipeline.predict(X_test)
catboost_metrics = {
    'mae': round(mean_absolute_error(y_test, catboost_pred), 2),
    'rmse': round(mean_squared_error(y_test, catboost_pred) ** 0.5, 2),
    'mape': round(mean_absolute_percentage_error(y_test, catboost_pred), 4),
    'r2': round(r2_score(y_test, catboost_pred), 4),
}
catboost_metrics

In [ ]:
# обе модели на одной тестовой выборке
pd.DataFrame({'бейзлайн (медиана)': baseline_metrics, 'catboost': catboost_metrics}).T

## 8. Важность признаков

Смотрю, на что модель опирается сильнее всего. Если бы в верхних строчках оказался какой-нибудь
идентификатор или колонка, которая по смыслу не влияет на цену, это был бы повод ещё раз проверить данные.

In [ ]:
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = pipeline.named_steps['model'].get_feature_importance()

pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

## Выводы

- CatBoost заметно точнее бейзлайна по всем четырём метрикам: MAE и RMSE ниже, R2 сильно выше нуля.
  Значит, признаки действительно объясняют цену, и модель имеет смысл.
- Основной метрикой оставляю MAE: она в рублях, устойчива к нескольким очень дорогим квартирам
  и понятна без пояснений. RMSE, MAPE и R2 считаю как дополнительные.
- В топе важности - площадь квартиры, координаты дома и год постройки. Это совпадает со здравым
  смыслом, подозрительных признаков в верхушке нет.

Дальше этот же код разложен по шагам пайплайна:

- `scripts/data.py` - выгружает `clean_flats_dataset` в `data/initial_data.csv`;
- `scripts/split.py` - делит данные на `data/train.csv` и `data/test.csv`;
- `scripts/fit.py` - обучает пайплайн и сохраняет `models/fitted_model.pkl`;
- `scripts/evaluate.py` - считает метрики на кросс-валидации и на тесте, результат пишет
  в `cv_results/cv_res.json`.

Параметры вынесены в `params.yaml`, порядок шагов и зависимости описаны в `dvc.yaml`.